In [1]:
import pandas as pd

In [2]:
chws_raw = pd.read_csv("../data/raw/community_health_workers_who.csv")
physicians_raw = pd.read_csv("../data/raw/physicians_who.csv")
nurses_and_midwives_raw = pd.read_csv("../data/raw/nurses_and_midwives_who.csv")

In [3]:
chws = chws_raw.sort_values(['SpatialDimValueCode', 'Period'], ascending=[True, False]).groupby('SpatialDimValueCode')['Value'].first().rename('chws')
chws = chws.str.replace(' ', '').astype(float)

physicians_raw['Indicator'].value_counts()
physicians = physicians_raw[physicians_raw['Indicator'] == 'Medical doctors (number)']
physicians = physicians.sort_values(['SpatialDimValueCode', 'Period'], ascending=[True, False]).groupby('SpatialDimValueCode')['Value'].first().rename('physicians')
physicians = physicians.str.replace(' ', '').astype(float)
physicians

SpatialDimValueCode
AFG    13155.0
AGO     8693.0
ALB     5397.0
AND      410.0
ARE    31844.0
        ...   
WSM      120.0
YEM     3872.0
ZAF    49533.0
ZMB     6531.0
ZWE     2222.0
Name: physicians, Length: 194, dtype: float64

In [4]:
nurses_and_midwives_raw['Indicator'].value_counts()
nurses_and_midwives = nurses_and_midwives_raw[nurses_and_midwives_raw['Indicator'].str.contains('(number)')]
nurses_and_midwives = nurses_and_midwives.sort_values(['SpatialDimValueCode', 'Period'], ascending=[True, False]).groupby('SpatialDimValueCode')['Value'].first().rename('nurse_and_midwives')
nurses_and_midwives = nurses_and_midwives.str.replace(' ', '').astype(float)
nurses_and_midwives

C:\Users\squaade\AppData\Local\Temp\ipykernel_45756\3243739200.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  nurses_and_midwives = nurses_and_midwives_raw[nurses_and_midwives_raw['Indicator'].str.contains('(number)')]


SpatialDimValueCode
AFG     22771.0
AGO     66751.0
ALB     16717.0
AND       382.0
ARE     63366.0
         ...   
WSM       648.0
YEM     21310.0
ZAF    398804.0
ZMB     59177.0
ZWE     49342.0
Name: nurse_and_midwives, Length: 194, dtype: float64

In [5]:
country_groups = pd.read_excel("../data/raw/CLASS_2025_10_07.xlsx")
ida_ibrd = country_groups[country_groups['Lending category'].notna()]

In [6]:
ida_ibrd['ida'] = ida_ibrd['Lending category'].isin(["IDA", "Blend"])
ida_ibrd['ibrd'] = ida_ibrd['Lending category'].isin(["IBRD", "Blend"])

ida_ibrd = ida_ibrd.set_index('Code')

In [13]:
health_workers = physicians.to_frame().join(chws).join(nurses_and_midwives)
health_workers['total'] = health_workers.sum(axis=1, skipna=True)
health_workers.index.rename('Code', inplace=True)

health_workers = health_workers.join(ida_ibrd[['ida', 'ibrd']])
health_workers_groups = health_workers[health_workers['ida'].notna()]
health_workers_groups.head(30)

,physicians,chws,nurse_and_midwives,total,ida,ibrd
Code,,,,,,
AFG,13155.0,29600.0,22771.0,65526.0,True,False
AGO,8693.0,7.0,66751.0,75451.0,False,True
ALB,5397.0,NaN,16717.0,22114.0,False,True
ARG,232525.0,NaN,216261.0,448786.0,False,True
ARM,9679.0,NaN,13675.0,23354.0,False,True
ATG,264.0,34.0,882.0,1180.0,False,True
AZE,32787.0,NaN,57425.0,90212.0,False,True
BDI,1038.0,11845.0,10873.0,23756.0,True,False
BEN,3042.0,11875.0,8616.0,23533.0,True,False


In [12]:
print(f"Total IDA health workers: {health_workers_groups[health_workers_groups['ida']]['total'].sum() / 1e6 :.0f}")
print(f"Total IBRD health workers: {health_workers_groups[health_workers_groups['ibrd']]['total'].sum() / 1e6:.0f}")
print(f"Total IDA and IBRD health workers (millions): {health_workers_groups['total'].sum() / 1e6:.0f}")

Total IDA health workers: 5
Total IBRD health workers: 30
Total IDA and IBRD health workers (millions): 34
